# 032 — Evaluation: EfficientNet NLL (Gaussian + beta)

Quantitative comparison of `efficientnet_unet_nll` on the held-out test set, trained with **both** Gaussian NLL variants (`022_training_efficientnet.ipynb`) — `gaussian_nll` and `beta_nll` — since this architecture still trains with both until Round 2 collapses it to one Laplace-beta loss, the way `031_evaluation_nll.ipynb`'s three architectures already were (`fixing.md` #10). Same block layout as `030_evaluation.ipynb` — see its title cell for the full table of companion notebooks.

Metrics (`mae`/`ssim`/`psnr`) are computed from the `mu` channel only (`scripts.metrics.Mu*Metric`); `loss` is the Gaussian NLL itself, not `combined_loss_advanced`, so it is *not* comparable in scale to `030`'s `loss` column.

Make the project root importable so `scripts.*` resolves regardless of
the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.trainer_nll import load_model_nll

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Build the test dataset

Artwork-and-mockups split, same as the notebook that trained these
checkpoints — required so the reported test metrics are on data none of
them saw during training or validation.

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
_, _, test_pairs = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

test_ds = build_dataset(
    test_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)
print(f"Test pairs: {len(test_pairs)} | Test batches: {len(test_ds)}")

## 2. Evaluate both loss variants

Checkpoints load from `models/nll_gaussian/efficientnet_unet_nll/best_model.keras`
and `models/nll_beta/efficientnet_unet_nll/best_model.keras`
(`022_training_efficientnet.ipynb`).

In [ ]:
ARCH = "efficientnet_unet_nll"
NLL_LOSSES = {"gaussian_nll": "nll_gaussian", "beta_nll": "nll_beta"}
results: dict = {}

for loss_name, tree in NLL_LOSSES.items():
    model_dir = settings.MODELS_DIR / tree
    try:
        model = load_model_nll(
            ARCH, model_dir=model_dir, loss_name=loss_name, beta=settings.NLL_BETA
        )
    except FileNotFoundError as exc:
        print(f"[skip] {exc}")
        continue

    print(f"Evaluating {ARCH} ({loss_name})...")
    metrics = model.evaluate(test_ds, verbose=0, return_dict=True)
    results[loss_name] = metrics
    print(f"  {loss_name}: { {k: f'{v:.4f}' for k, v in metrics.items()} }")

## 3. Comparison table

In [ ]:
if results:
    col_w = 24
    headers = ["loss"] + list(next(iter(results.values())).keys())
    print("".join(h.ljust(col_w) for h in headers))
    print("-" * (col_w * len(headers)))
    for loss_name, metrics in results.items():
        row = [loss_name] + [f"{v:.4f}" for v in metrics.values()]
        print("".join(c.ljust(col_w) for c in row))

## 4. Bar chart comparison

In [ ]:
if results:
    metric_keys = list(next(iter(results.values())).keys())
    n = len(metric_keys)
    loss_names = list(results.keys())
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(f"{ARCH} — gaussian_nll vs. beta_nll — test set")
    for ax, key in zip(axes, metric_keys):
        vals = [results[m][key] for m in loss_names]
        bars = ax.bar(loss_names, vals)
        ax.set_title(key.upper())
        ax.set_xticklabels(loss_names, rotation=20, ha="right")
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{v:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    plt.tight_layout()
    plt.show()